<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/ADPs/Lecture_6/%D0%9F%D1%80%D0%B0%D0%BA%D1%82%D0%B8%D1%87%D0%B5%D1%81%D0%BA%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическая работа № 6: Обучение моделей — от теории к практике

## Введение

В лекции №6 мы разобрали, как происходит обучение моделей машинного обучения: что такое функция потерь и градиентный спуск, чем отличаются параметры от гиперпараметров, зачем нужно разделение данных на train/validation/test, как интерпретировать графики обучения и какие метрики использовать для оценки качества модели в клинических задачах.

Теперь вам предстоит применить эти знания на практике.

**Цель работы** — закрепить навыки:
- создания и обработки синтетических данных для обучения,
- разделения данных на обучающую, валидационную и тестовую выборки,
- дообучения (fine-tuning) модели BERT на задачу классификации эмоций,
- построения и интерпретации графиков обучения,
- расчёта и интерпретации метрик качества (accuracy, precision, recall, F1),
- анализа матрицы ошибок,
- критической оценки результатов обучения с точки зрения клинической психологии.

---

## Подготовка рабочей среды

Перед началом работы установите необходимые библиотеки (в терминале или командной строке):

```bash
pip install transformers datasets scikit-learn matplotlib seaborn pandas numpy
```

Для работы с моделью ruBERT потребуется **около 2 ГБ свободного места** на диске. Если у вас ограниченные ресурсы, вы можете использовать более лёгкую модель `cointegrated/rubert-tiny`.

Импортируйте необходимые модули:




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)


## Часть 1. Теоретические вопросы (для самопроверки)

Перед выполнением практических заданий письменно ответьте на следующие вопросы. Это поможет убедиться, что вы понимаете ключевые концепции.

1. Что такое функция потерь (loss function)? Приведите аналогию из психотерапии.

2. Что такое градиентный спуск? Объясните на примере спуска с горы.

3. В чём разница между параметрами и гиперпараметрами модели? Приведите примеры тех и других.

4. Зачем делить данные на train, validation и test? Что произойдёт, если использовать test для настройки гиперпараметров?

5. Что такое утечка данных (data leakage)? Приведите пример в контексте психологического исследования.

6. Как выглядит график обучения при переобучении (overfitting)? Что нужно сделать, чтобы исправить ситуацию?

7. Как выглядит график обучения при недообучении (underfitting)? Что нужно сделать, чтобы исправить ситуацию?

8. Что такое precision и recall? Какая метрика важнее для скрининга депрессии и почему?

9. Что такое F1-score и когда его используют?

10. Зачем создают синтетические данные в клинической психологии? Каковы этические преимущества и ограничения?

11. Что такое дообучение (fine-tuning)? Почему оно эффективно для клинических задач и какие гиперпараметры критически важны?

12. **Этический вопрос:** представьте, что модель для выявления суицидального риска показывает precision = 90% и recall = 60%. Можно ли её использовать в клинике? Почему?





## Часть 2. Практические задания на Python

Все задания выполняйте в Jupyter Notebook. Код должен быть снабжён комментариями на русском языке. В конце каждого задания приводите краткий анализ результатов с точки зрения психолога.

---

### Задание 1. Создание и визуализация синтетического датасета эмоций

**Описание.** В этом задании вы создадите синтетический датасет, размеченный по четырём эмоциональным классам (гнев, страх, радость, грусть). Это имитирует сбор данных, который в реальной клинической практике потребовал бы этического разрешения и анонимизации.

**Требуется:**

1. Создайте датасет из 120 примеров (по 30 на каждый класс), используя приведённые ниже списки фраз.

2. Визуализируйте распределение классов с помощью столбчатой диаграммы.

3. Выведите несколько случайных примеров из датасета, чтобы убедиться, что данные корректны.

4. Вычислите и выведите среднюю длину текста (в символах) для каждого класса.


In [ ]:
# Ваш код решения задачи:



### Задание 2. Разделение данных на train, validation, test

**Описание.** В этом задании вы примените принципы разделения данных, описанные в лекции. Вы разделите датасет на обучающую (70%), валидационную (15%) и тестовую (15%) выборки с сохранением пропорции классов (стратификацией).

**Требуется:**

1. Разделите данные на train (70%), validation (15%), test (15%) с использованием `train_test_split`.

2. Используйте `stratify=y` для сохранения пропорции классов.

3. Выведите размер каждой выборки и проверьте, что пропорции классов сохранены.

4. Сохраните размеченные данные в CSV-файл с колонкой `split`, указывающей принадлежность к выборке.




In [ ]:
# Ваш код решения задачи:

### Задание 3. Токенизация и загрузка модели

**Описание.** В этом задании вы подготовите данные для обучения: преобразуете тексты в числовые представления (токены) и загрузите предобученную модель ruBERT.

**Требуется:**

1. Загрузите токенизатор для модели `DeepPavlov/rubert-base-cased` (или `cointegrated/rubert-tiny`, если ограничены ресурсы).

2. Напишите функцию токенизации, которая преобразует тексты в токены с максимальной длиной 64.

3. Примените токенизацию ко всем трём выборкам (train, validation, test).

4. Загрузите модель `AutoModelForSequenceClassification` с 4 классами.

5. Выведите количество параметров модели.





In [ ]:
# Ваш код решения задачи:

### Задание 4. Дообучение модели и мониторинг графиков

**Описание.** Теперь вы выполните дообучение (fine-tuning) модели на задачу классификации эмоций. Это основная часть работы, где вы увидите все процессы, описанные в лекции: снижение loss, рост accuracy, возможное переобучение.

**Требуется:**

1. Настройте гиперпараметры обучения:
   - `learning_rate = 2e-5`
   - `num_train_epochs = 5`
   - `per_device_train_batch_size = 8`
   - `weight_decay = 0.01`
   - `eval_strategy = "epoch"`
   - `save_strategy = "epoch"`
   - `load_best_model_at_end = True`

2. Создайте `Trainer` с функцией для расчёта метрик (accuracy, precision, recall, f1).

3. Добавьте `EarlyStoppingCallback` с `early_stopping_patience = 2`.

4. Запустите обучение (это может занять 10–30 минут в зависимости от ресурсов).

5. Сохраните историю обучения (логи) для построения графиков.



In [ ]:
# Ваш код решения задачи:

### Задание 5. Построение и интерпретация графиков обучения

**Описание.** После завершения обучения вы построите графики loss и accuracy и проанализируете, как прошло обучение.

**Требуется:**

1. Извлеките из истории обучения значения train loss, validation loss и validation accuracy на каждом шаге/эпохе.

2. Постройте два графика на одной фигуре:
   - Слева: динамика loss (train и validation).
   - Справа: динамика accuracy (validation).

3. Оформите графики по правилам APA: заголовки, подписи осей, легенда.

4. Напишите анализ (5–7 предложений):
   - Снижался ли loss на обучающей и валидационной выборках?
   - Был ли разрыв между train и val кривыми?
   - Наблюдалось ли переобучение или недообучение?
   - Какой момент был оптимальным для остановки обучения?





In [ ]:
# Ваш код решения задачи:

### Задание 6. Оценка модели на тестовой выборке и расчёт метрик

**Описание.** В этом задании вы оцените финальную модель на тестовой выборке и рассчитаете все ключевые метрики.

**Требуется:**

1. Загрузите лучшую модель (она сохранится автоматически благодаря `load_best_model_at_end`).

2. Получите предсказания на тестовой выборке с помощью `trainer.predict()`.

3. Рассчитайте:
   - Accuracy
   - Precision (weighted)
   - Recall (weighted)
   - F1-score (weighted)

4. Выведите classification report для каждого класса.

5. Постройте матрицу ошибок (confusion matrix) в виде тепловой карты.

6. Напишите интерпретацию результатов (5–7 предложений):
   - Какие классы модель распознаёт лучше всего?
   - Какие классы модель путает чаще всего?
   - Каков клинический смысл этих ошибок?
   - Достаточно ли высоки метрики для использования в психологической практике?





In [ ]:
# Ваш код решения задачи:

### Задание 7. Эксперимент с гиперпараметрами (сравнительный анализ)

**Описание.** В этом задании вы проведёте небольшой эксперимент: обучите модель с другими гиперпараметрами и сравните результаты.

**Требуется:**

1. Обучите модель с **увеличенным learning rate** (`5e-5`) на 5 эпох.

2. Обучите модель с **уменьшенным числом эпох** (`2 эпохи`).

3. Сравните результаты всех трёх экспериментов (исходный, с увеличенным LR, с уменьшенными эпохами) по метрикам.

4. Напишите вывод: какие гиперпараметры дали лучший результат и почему?




In [ ]:
# Ваш код решения задачи:

### Задание 8. Этический анализ: модель для клинической практики

**Описание.** Представьте, что вы — клинический психолог и руководитель центра психического здоровья. Ваша дообученная модель показывает следующие результаты на тестовой выборке:

| Класс | Precision | Recall | F1-score |
|-------|-----------|--------|----------|
| Гнев | 0.82 | 0.78 | 0.80 |
| Страх | 0.79 | 0.81 | 0.80 |
| Радость | 0.88 | 0.85 | 0.86 |
| Грусть | 0.76 | 0.80 | 0.78 |
| **Среднее (weighted)** | **0.81** | **0.81** | **0.81** |

**Напишите этический протокол** (2–3 страницы), в котором осветите:

1. **Клиническая целесообразность** — для какой задачи можно использовать эту модель? Для каких задач использовать нельзя? Почему?

2. **Информированное согласие** — что пациент должен знать об использовании модели?

3. **Человеческий контроль** — как организовать проверку прогнозов модели психологом?

4. **Обработка ошибок** — что делать при ложных срабатываниях (FP) и пропущенных случаях (FN)?

5. **Мониторинг качества** — как часто переоценивать модель на новых данных?

6. **Прозрачность** — как объяснить пациенту, почему модель выдала тот или иной прогноз?





```
# Ваш анализ:
```





## Часть 3. Комплексное задание (повышенной сложности) — по желанию

**Задание 9. Дообучение на реальных данных (Hugging Face Datasets)**

**Описание.** Найдите на Hugging Face датасет для классификации эмоций или тональности на русском языке (например, `ru-goemotions`, `linanqiu/ru_emotion` или другой). Загрузите его и выполните дообучение модели.

**Требуется:**

1. Загрузите датасет с Hugging Face с помощью `datasets.load_dataset()`.

2. Изучите структуру датасета: сколько примеров, какие классы, распределение классов.

3. Выполните все шаги, как в заданиях 2–6, но на реальных данных.

4. Сравните результаты с синтетическим датасетом. Что изменилось? Почему?

5. Напишите рефлексию (1 страница): с какими сложностями вы столкнулись при работе с реальными данными?





In [ ]:
# Ваш код решения задачи:

## Критерии оценки

| Компонент | Процент | Описание |
|-----------|---------|----------|
| Теоретические вопросы (Часть 1) | 15% | Полнота и правильность ответов |
| Задание 1 (создание датасета) | 5% | Корректность кода, визуализация |
| Задание 2 (разделение данных) | 5% | Правильность разделения, стратификация |
| Задание 3 (токенизация) | 5% | Корректность токенизации |
| Задание 4 (дообучение) | 15% | Корректность запуска, сохранение логов |
| Задание 5 (графики) | 15% | Качество графиков, глубина интерпретации |
| Задание 6 (метрики) | 15% | Корректность расчётов, интерпретация |
| Задание 7 (эксперимент) | 10% | Сравнительный анализ, выводы |
| Задание 8 (этический протокол) | 15% | Полнота, аргументированность, практичность |
| Задание 9 (бонус, по желанию) | +5% | Корректность, рефлексия |

---

## Требования к сдаче

- Пришлите **один файл** (Jupyter Notebook `.ipynb`) со всеми заданиями, кодом и текстовыми комментариями.
- Все графики должны быть оформлены аккуратно: подписи осей, легенды, заголовки.
- Этический протокол (Задание 8) приложите отдельным файлом (`.txt` или `.pdf`) или в виде текстовой ячейки в Notebook.
- Убедитесь, что код выполняется без ошибок. Укажите версии библиотек при необходимости.
- Все результаты анализа должны сопровождаться интерпретацией с точки зрения психолога.

---

## Заключение

Данная практическая работа проведёт вас через полный цикл дообучения NLP-модели — от создания синтетических данных до оценки качества и этического анализа. Вы не только освоите технические навыки, но и научитесь **критически интерпретировать** результаты обучения и принимать обоснованные решения о применении моделей в клинической практике.

**Главный вывод:** понимание процесса обучения позволяет клиницисту задавать правильные вопросы: *«На каких данных обучена модель? Как она делила выборку? Не переобучилась ли она? Какую метрику она оптимизировала?»* Это знание — основа ответственного использования ИИ в психологии.

---

**Срок выполнения: 2 недели.**